In [ ]:
from torch.nn import *
import numpy as np
import torch

torch.set_default_tensor_type(torch.cuda.FloatTensor)


def unpack(a, n, m=2 ** torch.arange(16)):
    return (a[..., None] & m[:n]).bool().float()


def dat(k, c, len=0, bs=4, std=256, code="bin"):
    len = 2 ** int(np.log2((len or k + 1) * c))
    y = torch.randint(c, (bs, len))
    x = torch.randn(bs, len, k).add_(torch.randn(bs, c, k).mul_(std)[torch.arange(bs)[:, None], y])
    if code == "onehot":
        return torch.cat((x, functional.one_hot(y, c).float()), -1), x, y
    y = unpack(y, np.log2(c).__ceil__())
    return torch.cat((x, y), -1), x, y


def last_eval(m, d, x, y):
    w, W = m[0].weight.T.split((d.size(2), x.size(2)), 0)
    return m[1:](d @ w + x @ W), y


def los(x, y, code="bin"):
    if code == "onehot":
        return functional.cross_entropy(x.transpose(1, 2), y)
    return functional.binary_cross_entropy_with_logits(x, y.broadcast_to(x.shape))


def acc(x, y, code="bin"):
    if code == "onehot":
        x = x.argmax(-1) == y
    else:
        x = (x.signbit() != y).all(-1)
    return x.sum() / x.numel()


class BatchNorm(BatchNorm1d):
    def forward(s, x):
        return super().forward(x.view(-1, x.size(-1))).view(x.shape)


def Nn(i, o=0, m=8, b=1, bn=1, act=Hardswish(1)):
    o = o or i
    m = max(i, o, m)
    return Sequential(Linear(i, m, not bn), *(BatchNorm1d(m, track_running_stats=0),) if bn else (), act, Linear(m, o, b)) if act else Linear(i, o, b)


class Tnn(Sequential):
    def __init__(s, *m, sc=0):
        super().__init__(*m)
        s.sc = sc

    def forward(s, x):
        for _ in range(int(np.log2(x.size(1)))):
            _ = super().forward(x.view(-1, 2 * x.size(-1)))
            x = x.view(-1, 2, x.size(-1)).mean(1).add_(_) if s.sc else _
        return x


class TruSimple(Sequential):
    def forward(s, x):
        k = super().forward(x).sigmoid_()
        x = x.view(-1, 2, x.size(-1) // 2)
        return (x[:, 0] * k).add_(x[:, 1] * (1 - k))


class Tru(Module):
    def __init__(s, forget, In, out):
        super().__init__()
        s.forget = forget
        s.In = In
        s.out = out

    def forward(s, x):
        x = torch.cat((x, s.In(x * s.forget(x).sigmoid_())), 1)
        return (x.view(-1, 3, x.size(1) // 3) * s.out(x).view(-1, 3, x.size(1) // 3).softmax(1)).sum(1)


class SelfAttentionNn(Module):
    def __init__(s, forget, In, out):
        super().__init__()
        s.forget = forget
        s.In = In
        s.out = out

    def forward(s, x):
        y = s.In(x * s.forget(x).sigmoid_())
        w = s.out(torch.cat((x, y), 1)).sigmoid_()
        return (x * w).add_(y.mul_(1 - w))


class SelfMultiAttentionNn(SelfAttentionNn):
    def forward(s, x):
        x = x, s.In(x * s.forget(x).sigmoid_())
        n = x[1].size(-1)
        x = torch.cat(x, 1)
        return (x.view(-1, x.size(-1) // n, n) * s.out(x).view(-1, x.size(-1) // n, n).softmax(-2)).sum(-2)

    @classmethod
    def of(s, k, fac=1, n=2):
        return s(Nn(k * 2, k * 2, k * fac), Nn(k * 2, k, k * fac), Nn(k * 3, k * 3, k * fac))


def model(
    k,
    c,
    fac=1,
    exp=1,
    midfac=0,
    midexp=0,
    kind=SelfMultiAttentionNn,
    b0=1,
    bn0=0,
    act0=Hardswish(1),
    b1=1,
    bn1=1,
    act1=Hardswish(1),
    b=1,
    bn=1,
    act=Hardswish(1),
    b2=1,
    bn2=1,
    act2=Hardswish(1),
    b3=1,
    bn3=0,
    act3=Hardswish(1),
    code="bin",
    **_
):
    k1 = int((k * c) ** exp * fac)
    k2 = int((k * c) ** (midexp or exp) * (midfac or fac))
    if code != "onehot":
        c = np.log2(c).__ceil__()
    M = Sequential(
        Nn(k + c, k1, k2, b0, bn0, act0),
        Tnn(
            kind(Nn(k1 * 2, k1, k2, b, bn, act))
            if kind == TruSimple
            else kind(Nn(k1 * 2, k1 * 2, k2, b1, bn1, act1), Nn(k1 * 2, k1, k2, b, bn, act), Nn(k1 * 3, k1 * 3, k2, b2, bn2, act2))
            if kind == Tru
            else SelfMultiAttentionNn(Nn(k1 * 2, k1 * 2, k2, b1, bn1, act1), Nn(k1 * 2, k1, k2, b, bn, act), Nn(k1 * 3, k1 * 3, k2, b2, bn2, act2))
            if kind == SelfMultiAttentionNn
            else kind(k1 * 2, k1, k2, b, bn, act),
            **_
        ),
    )
    m = Nn(k1 + k, c, k2, b3, bn3, act3)
    return M, m

In [ ]:
k = 2
c = 64
M, m = model(k, c)
o = torch.optim.Adam((*M.parameters(), *m.parameters()))
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.002, 100000, pct_start=0.01, div_factor=9, final_div_factor=999)
s = torch.cuda.amp.GradScaler()
for i in range(O.total_steps // 1000):
    l = []
    a = []
    for i in range(1000):
        o.zero_grad()
        with torch.cuda.amp.autocast():
            i = dat(k, c, bs=8, len=1)
            i = last_eval(m, M(i[0])[:, None], *i[1:])
            a.append(acc(*i))
            i = los(*i)
        s.scale(i).backward()
        s.step(o)
        s.update()
        O.step()
        l.append(i)
    print(sum(l) / len(l), sum(a) / len(a), o.param_groups[0]["lr"])